In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch06-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "06-01-molecular-fingerprints.py"

比較 ESOL 裡三種單取代苯的路徑指紋。首次執行需連網下載 ESOL。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile

import deepchem as dc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import Draw, rdFingerprintGenerator

# ESOL：1,128 筆，實測 log10 溶解度（mol/L），Delaney DOI 10.1021/ci034243x。
# 這裡只讀結構；保留原始標籤，沒有訓練模型。
np.random.seed(42)
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "cheminfo-datasets"))
cache.mkdir(parents=True, exist_ok=True)
tasks, splits, _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="scaffold",
    transformers=[], data_dir=str(cache), reload=False)
assert sum(map(len, splits)) == 1128
records = {str(s).strip() for split in splits for s in split.ids}
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)

examples = {"Phenol": "c1ccccc1O", "Anisole": "COc1ccccc1", "Ethylbenzene": "CCc1ccccc1"}
assert set(examples.values()) <= records
mols = {name: Chem.MolFromSmiles(smiles) for name, smiles in examples.items()}
generator = rdFingerprintGenerator.GetRDKitFPGenerator(
    minPath=1, maxPath=3, fpSize=128, numBitsPerFeature=1)
bits = {name: set(generator.GetFingerprint(mol).GetOnBits())
        for name, mol in mols.items()}
for name, active in bits.items():
    print(name, len(active), sorted(active))

from matplotlib.colors import ListedColormap
columns = sorted(set.union(*bits.values()))
fig = plt.figure(figsize=(8.6, 4.7), layout="constrained")
grid = fig.add_gridspec(2, 3, height_ratios=[1.4, 1])
for i, (name, mol) in enumerate(mols.items()):
    ax = fig.add_subplot(grid[0, i])
    ax.imshow(Draw.MolToImage(mol, size=(300, 170)))
    ax.set_title(name)
    ax.axis("off")
ax = fig.add_subplot(grid[1, :])
matrix = [[j in bits[name] for j in columns] for name in mols]
ax.imshow(matrix, cmap=ListedColormap(["#eef1f2", "#136b71"]), aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(columns)), columns)
ax.set_yticks(range(len(mols)), list(mols))
ax.set(xlabel="Bit index (columns with at least one hit)")
fig.savefig(figure_dir / "06-01-molecular-fingerprints.png", dpi=150)

# 檢查各分子裡實際觸發這些位元的鍵，不把欄號當官能基名稱。
for name, mol in mols.items():
    info = rdFingerprintGenerator.AdditionalOutput()
    info.CollectBitPaths()
    generator.GetFingerprint(mol, additionalOutput=info)
    for index in (33, 44, 120):
        paths = info.GetBitPaths().get(index, ())
        fragments = set()
        for path in paths:
            atoms = set()
            for bond_index in path:
                bond = mol.GetBondWithIdx(bond_index)
                atoms.update([bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()])
            fragments.add(Chem.MolFragmentToSmiles(mol, atomsToUse=sorted(atoms),
                                                   bondsToUse=list(path)))
        print(name, index, len(paths), sorted(fragments))


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "06-01-molecular-fingerprints.png")))